In [ ]:
import os
from collections import Counter
import numpy as np
import matplotlib.pyplot as plt
from wordcloud import WordCloud
import platform

# ==================== 1. 配置路径 ====================
# 根据截图修改为人类基因所在的文件夹路径 (WSL格式)
INPUT_DIR = "/mnt/e/2-8.3-shanda/1-feature/1-human-guaidian-choose-gene/Gene_Lists"
# 输出路径，建议新建一个专门放人类词云图的文件夹
OUTPUT_DIR = "/mnt/e/2-8.3-shanda/1-feature/1-figure/1-1-result-1-human-gene/1-human-ciyuntu"

# --- 核心修复：智能识别 WSL 环境并读取 Arial 字体 ---
system = platform.system()
if system == "Windows":
    FONT_PATH = "C:/Windows/Fonts/arial.ttf"
elif system == "Darwin": # macOS
    FONT_PATH = "/Library/Fonts/Arial.ttf"
else: # Linux / WSL
    # 优先尝试在 WSL 中直接访问 Windows 宿主机的字体
    wsl_font_path = "/mnt/c/Windows/Fonts/arial.ttf"
    linux_native_path = "/usr/share/fonts/truetype/msttcorefonts/Arial.ttf"
    
    if os.path.exists(wsl_font_path):
        FONT_PATH = wsl_font_path
    elif os.path.exists(linux_native_path):
        FONT_PATH = linux_native_path
    else:
        # 终极兜底：如果你不在 WSL 且没装字体，请把 arial.ttf 拷到代码运行的当前目录
        print("⚠️ 未能在系统路径中找到 Arial 字体。正在尝试使用当前目录下的 arial.ttf...")
        FONT_PATH = "arial.ttf" 

# ==================== 2. 主刊级别视觉配置 ====================
COLOR_MAP = 'plasma'
MAX_WORDS = 100 
DPI = 300

# ==================== 3. 核心处理逻辑 ====================
os.makedirs(OUTPUT_DIR, exist_ok=True)

def create_oval_mask(width=2000, height=1200):
    """生成椭圆形的掩模阵列"""
    x, y = np.ogrid[:height, :width]
    center_x, center_y = width / 2, height / 2
    # 椭圆方程
    mask = ((x - center_y) ** 2 / (height * 0.45) ** 2 +
            (y - center_x) ** 2 / (width * 0.45) ** 2) > 1
    return 255 * mask.astype(int)

if not os.path.exists(INPUT_DIR):
    print(f"错误: 找不到目录 {INPUT_DIR}")
else:
    files = [f for f in os.listdir(INPUT_DIR) if f.endswith('.txt')]
    all_genes = []

    for filename in files:
        file_path = os.path.join(INPUT_DIR, filename)
        with open(file_path, 'r') as f:
            # 【核心修改点】：保证人类基因命名规范 (全部大写)
            all_genes.extend([line.strip().upper() for line in f if line.strip()])

    if not all_genes:
        print("未找到基因数据。请检查文本文件中是否有内容。")
    else:
        gene_freq = dict(Counter(all_genes))

        print(f"正在生成人类基因主刊级椭圆词云 (Color-safe: {COLOR_MAP})...")

        oval_mask = create_oval_mask(2000, 1200) 

        try:
            wc = WordCloud(
                font_path=FONT_PATH,    # 使用修复后的 Arial 路径
                width=2000, height=1200,
                background_color='white',
                mask=oval_mask,
                colormap=COLOR_MAP,
                max_words=MAX_WORDS,
                min_font_size=12,
                max_font_size=250,      
                random_state=42,
                prefer_horizontal=1.0,  
                relative_scaling=0.5,   
                contour_width=0,        
            )

            wc.generate_from_frequencies(gene_freq)

            # --- 绘图展示 (双栏物理尺寸 7英寸 x 4.2英寸) ---
            plt.figure(figsize=(7, 4.2))
            plt.imshow(wc, interpolation="bilinear")
            plt.axis("off")

            # --- 保存出版级文件 ---
            # 修改了输出文件的前缀名
            out_base = os.path.join(OUTPUT_DIR, "Human_Gene_Cloud_Oval")
            
            # 1. 保存高分 PNG 预览
            plt.savefig(f"{out_base}.png", dpi=DPI, format='png', bbox_inches='tight', pad_inches=0.1)
            
            # 2. 保存包裹图片的 PDF
            plt.savefig(f"{out_base}.pdf", dpi=DPI, format='pdf', bbox_inches='tight', pad_inches=0.1)
            
            # 3. 保存纯文本矢量 SVG (强烈推荐用于 Adobe Illustrator 排版)
            svg_text = wc.to_svg(embed_font=True)
            with open(f"{out_base}.svg", "w", encoding="utf-8") as f:
                f.write(svg_text)

            plt.show()

            print("\n" + "="*40)
            print(f"✅ 处理完成！")
            print(f"1. 基因已规范化为人类格式 (全部大写，如 SOX2)")
            print(f"2. 词云已保存为 PNG, PDF 和 纯矢量 SVG 格式，路径：{OUTPUT_DIR}")
            print(f"3. 字体成功调用: {FONT_PATH}")
            print("="*40)
            
        except OSError as e:
            print("\n❌ 严重错误: 依然无法加载 Arial 字体。")
            print("请尝试以下终极解决方案：")
            print("1. 在 Windows 中打开 C:\\Windows\\Fonts")
            print("2. 复制 arial.ttf 文件到你的 Linux 代码目录")
            print("3. 再次运行本代码")

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
import re
import mygene

# ==========================================
# 1. 图表全局设置 (主刊级极简细线风格)
# ==========================================
mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['ps.fonttype'] = 42
mpl.rcParams['font.family'] = 'sans-serif'
mpl.rcParams['font.sans-serif'] = ['Arial', 'Helvetica']
# 主刊要求极细的坐标轴线条，推荐 0.5pt
mpl.rcParams['axes.linewidth'] = 0.5 

# 正则匹配辅助函数，避免无边界字符串的错误匹配
def match_keywords(text, keywords):
    for k in keywords:
        # \b 表示单词边界，re.escape 确保特殊字符安全解析
        if re.search(r'\b' + re.escape(k) + r'\b', text):
            return True
    return False

# ==========================================
# 2. 从联网数据库获取基因客观分类 (改为 Human)
# ==========================================
def get_objective_classifications(gene_list):
    if not gene_list:
        print("警告：未读取到任何基因，请检查 txt 文件路径。")
        return {}

    print(f"正在深度联网查询 {len(gene_list)} 个人类基因的名称与 GO 功能库，请稍候...")
    mg = mygene.MyGeneInfo()
    
    # 获取人类基因的 GO 注释
    results = mg.querymany(gene_list, scopes='symbol', fields='name,type_of_gene,go', species='human')

    gene_info_dict = {}
    for res in results:
        query_gene = res.get('query')
        if query_gene and query_gene not in gene_info_dict:
            gene_info_dict[query_gene] = res

    classification_dict = {}
    for gene in gene_list:
        try:
            row = gene_info_dict.get(gene, {})
            symbol = str(gene).upper()
            desc = str(row.get('name', '')).lower()
            gene_type = str(row.get('type_of_gene', '')).lower()

            # 提取 GO 注释并拼接
            go_terms = ""
            if 'go' in row:
                go_data = row['go']
                for sub_ont in ['BP', 'MF', 'CC']:  
                    if sub_ont in go_data:
                        items = go_data[sub_ont]
                        if isinstance(items, dict): items = [items]
                        if isinstance(items, list):
                            for item in items:
                                go_terms += str(item.get('term', '')).lower() + " "
            
            search_text = desc + " " + go_terms + " " + gene_type

            # --- 严格正则分类判断引擎 ---
            
            # 1. 过滤未定义基因与假基因 
            if 'rna' in gene_type or 'pseudo' in gene_type or re.match(r'^LOC\d+$', symbol) or re.match(r'^LINC\d+$', symbol) or 'ORF' in symbol or re.match(r'^GM\d+$', symbol) or symbol.endswith('RIK') or re.match(r'^BC\d+$', symbol):
                category = 'ncRNA & Uncharacterized'
            
            # 2. 【最高优先级】：核糖体、RNA加工与翻译
            elif symbol.startswith(('RPL', 'RPS', 'MRPL', 'MRPS', 'EIF', 'EEF', 'SNRP', 'HNRNP', 'DDX')) or \
                 match_keywords(search_text, ['ribosome', 'ribosomal', 'translation', 'trna', 'rrna', 'mrna', 'elongation factor', 'initiation factor', 'rna binding', 'spliceosome', 'splicing', 'rna processing', 'ribonucleoprotein']):
                category = 'Ribosome, RNA & Translation'
                
            # 3. 细胞骨架与 ECM
            elif match_keywords(search_text, ['collagen', 'matrix', 'actin', 'myosin', 'adhesion', 'elastin', 'cytoskeleton', 'keratin', 'tubulin', 'vimentin', 'integrin', 'laminin', 'fibronectin', 'cadherin', 'microtubule', 'extracellular']):
                category = 'Cytoskeleton & ECM'
                
            # 4. 免疫与防御
            elif match_keywords(search_text, ['immune', 'chemokine', 'interleukin', 'cd antigen', 'histocompatibility', 'complement', 'lysozyme', 's100', 'interferon', 'macrophage', 't cell', 'b cell', 'antigen', 'toll-like', 'defensin', 'leukocyte', 'inflammatory', 'mhc', 'phagocytosis']):
                category = 'Immunity & Defense'
                
            # 5. 细胞周期与凋亡
            elif match_keywords(search_text, ['cell cycle', 'apoptosis', 'cyclin', 'caspase', 'cell division', 'mitosis', 'meiosis', 'senescence', 'death', 'p53', 'proliferation']):
                category = 'Cell Cycle & Apoptosis'
                
            # 6. 蛋白折叠与降解
            elif match_keywords(search_text, ['ubiquitin', 'proteasome', 'chaperone', 'heat shock', 'autophagy', 'peptidase', 'protease', 'degradation', 'protein folding']):
                category = 'Protein Folding & Degradation'
                
            # 7. 表观遗传与染色质
            elif match_keywords(search_text, ['histone', 'chromatin', 'methyltransferase', 'acetyltransferase', 'epigenetic', 'nucleosome', 'hdac', 'dna methylation']):
                category = 'Epigenetics & Chromatin'
                
            # 8. 转录调控
            elif match_keywords(search_text, ['transcription', 'polymerase', 'zinc finger', 'box', 'promoter']):
                category = 'Transcription Regulation'
                
            # 9. 发育与分化
            elif match_keywords(search_text, ['development', 'differentiation', 'morphogenesis', 'homeobox', 'hox', 'embryonic', 'stem cell', 'angiogenesis', 'osteogenesis', 'chondrogenesis']):
                category = 'Development & Differentiation'
                
            # 10. 囊泡运输与细胞膜
            elif match_keywords(search_text, ['vesicle', 'exocytosis', 'endocytosis', 'rab', 'snare', 'clathrin', 'golgi', 'endoplasmic reticulum', 'membrane', 'peroxisome', 'lysosome', 'vacuole']):
                category = 'Vesicular Transport & Cytomembrane'
                
            # 11. 信号传导与受体
            elif match_keywords(search_text, ['kinase', 'phosphatase', 'receptor', 'signal', 'g-protein', 'ras', 'wnt', 'tgf', 'bmp', 'notch', 'hedgehog', 'hormone', 'calcium', 'camp']):
                category = 'Signaling & Receptors'
                
            # 12. 代谢与转运
            elif match_keywords(search_text, ['atp', 'cytochrome', 'metabolism', 'dehydrogenase', 'synthase', 'solute carrier', 'transporter', 'lipid', 'cholesterol', 'fatty acid', 'glycolysis', 'mitochondrial', 'channel', 'pump', 'oxidoreductase', 'catalase', 'transferase', 'reductase', 'catabolic']):
                category = 'Metabolism & Transport'
                
            # 13. 其他
            else:
                category = 'Other / Unclassified'

            classification_dict[gene] = category
        except Exception:
            classification_dict[gene] = 'Other / Unclassified'

    return classification_dict

# ==========================================
# 3. 路径分离配置与本地文件读取
# ==========================================
INPUT_DIR = "/mnt/e/2-8.3-shanda/1-feature/1-human-guaidian-choose-gene/Gene_Lists" 
OUTPUT_DIR = "/mnt/e/2-8.3-shanda/1-feature/1-figure/1-1-result-1-human-gene/2-ratio-human-gene-tissue"
os.makedirs(OUTPUT_DIR, exist_ok=True)

all_genes = set()
tissue_genes = {}

for file in os.listdir(INPUT_DIR):
    if file.endswith('.txt') and "_Knee_" in file and "_Genes" in file:
        
        # 🌟 核心修改点：分步精准剥离冗余信息
        # 1. 剥离尾部的 "_Knee_数字_Genes.txt"
        raw_name = re.sub(r'_Knee_\d+_Genes\.txt', '', file)
        
        # 2. 剥离头部的 "type_数字_" (忽略大小写以防万一)
        clean_name = re.sub(r'^type_\d+_', '', raw_name, flags=re.IGNORECASE)
        
        # 3. 把剩余可能的下划线替换为空格，形成最终纯净组织名
        tissue_name = clean_name.replace('_', ' ')
        
        with open(os.path.join(INPUT_DIR, file), 'r', encoding='utf-8') as f:
            genes = [line.strip().split()[-1].upper() for line in f.readlines() if line.strip()]
            tissue_genes[tissue_name] = genes
            all_genes.update(genes)

# ==========================================
# 4. 获取归类并计算比例
# ==========================================
global_gene_dict = get_objective_classifications(list(all_genes))

categories = [
    'Ribosome, RNA & Translation', 'Cytoskeleton & ECM', 'Immunity & Defense',
    'Cell Cycle & Apoptosis', 'Protein Folding & Degradation', 'Epigenetics & Chromatin', 
    'Transcription Regulation', 'Development & Differentiation', 
    'Vesicular Transport & Cytomembrane', 'Signaling & Receptors', 
    'Metabolism & Transport', 'ncRNA & Uncharacterized', 'Other / Unclassified'
]

results = []
for tissue, genes in tissue_genes.items():
    if not genes: continue
    counts = {cat: 0 for cat in categories}
    for g in genes:
        cat = global_gene_dict.get(g, 'Other / Unclassified')
        counts[cat] += 1

    total = sum(counts.values())
    perc = {cat: (counts[cat]/total)*100 for cat in categories}
    perc['Tissue'] = tissue
    results.append(perc)

df = pd.DataFrame(results)
if df.empty:
    print(f"未生成任何数据，可能是 {INPUT_DIR} 目录下没有找到符合包含 '_Knee_' 和 '_Genes' 的 txt 文件！")
else:
    df.set_index('Tissue', inplace=True)
    df = df.sort_values(by='Ribosome, RNA & Translation', ascending=False)
    
    df.to_csv(os.path.join(OUTPUT_DIR, "Human_Gene_Classification_Data.csv"))
    pd.DataFrame.from_dict(global_gene_dict, orient='index', columns=['Category']).to_csv(os.path.join(OUTPUT_DIR, "Human_Gene_to_Category_Map.csv"))

    # ==========================================
    # 5. 绘制主刊级堆叠柱状图 (终极抛光版)
    # ==========================================
    # --- 精准锁定物理尺寸: 126 mm x 60 mm ---
    width_in = 126 / 25.4   
    height_in = 60 / 25.4   
    fig, ax = plt.subplots(figsize=(width_in, height_in))

    # --- 坚持使用指定的 NPG 经典原版配色 ---
    colors = [
        '#E64B35', '#4DBBD5', '#00A087', '#3C5488', '#F39B7F', '#8491B4', 
        '#91D1C2', '#DC0000', '#7E6148', '#B09C85', '#9467BD', '#FF9896', '#E0E0E0'
    ]

    ax.yaxis.grid(True, color='#E6E6E6', linestyle='-', linewidth=0.5, zorder=0)

    # 堆叠图绘制
    df[categories].plot(kind='bar', stacked=True, ax=ax, color=colors, 
                        width=0.8, edgecolor='white', linewidth=0.5, zorder=3)

    ax.set_ylim(0, 100)
    
    # 细化 Y 轴标题设置
    ax.set_ylabel('Percentage of Genes (%)', fontsize=7, fontweight='normal', labelpad=2)
    ax.set_xlabel('', fontsize=7)
    
    plt.xticks(rotation=45, ha='right', fontsize=6)
    plt.yticks(fontsize=6)
    
    ax.tick_params(axis='y', which='major', width=0.5, length=2.5, pad=2)
    ax.tick_params(axis='x', which='major', width=0.5, length=2.5, pad=0)
    
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

    # 图例设置保持紧凑
    legend = ax.legend(loc='center left', bbox_to_anchor=(1.02, 0.5), fontsize=5.5, 
                       frameon=False, title='Gene Functional Class', title_fontsize=6.5, 
                       labelspacing=0.6, handlelength=1.0, handleheight=0.6)
    legend.get_title().set_fontweight('bold')

    plt.tight_layout()
    
    output_pdf = os.path.join(OUTPUT_DIR, "Human_Genes_Proportions_Final.pdf")
    output_png = os.path.join(OUTPUT_DIR, "Human_Genes_Proportions_Final.png")
    output_svg = os.path.join(OUTPUT_DIR, "Human_Genes_Proportions_Final.svg")
    
    plt.savefig(output_pdf, format='pdf', bbox_inches='tight', pad_inches=0.03, facecolor='white')
    plt.savefig(output_png, dpi=300, format='png', bbox_inches='tight', pad_inches=0.03, facecolor='white')
    plt.savefig(output_svg, format='svg', bbox_inches='tight', pad_inches=0.03, facecolor='white')

    print(f"\n✅ 处理完成！分类数据与可视化图表已保存至: {OUTPUT_DIR}")